## AND-102 Task 5: ETL Pipeline — Dataset Merging

## AND-102 Task 5: Merge 1 — License + Installed

In [1]:
import pandas as pd

license_df = pd.read_csv('../data/license.csv')
license_df = license_df[license_df['LICENSESTATUS'].isin(['ACTIVE', 'PENDING_RENEWAL'])]


In [2]:
installed_df = pd.read_json('../data/installed.json')
installed_df = installed_df.rename(columns={'Elevating devices number': 'ElevatingDevicesNumber'})


In [3]:
print(f'license rows (filtered): {len(license_df):,}')
print(f'installed rows:          {len(installed_df):,}')


license rows (filtered): 43,297
installed rows:          46,936


In [4]:
# Join key: ElevatingDevicesNumber (renamed from 'Elevating devices number' in installed.json)
merged_df = license_df.merge(installed_df, on='ElevatingDevicesNumber', how='inner')


In [5]:
lost = len(license_df) - len(merged_df)
print(f'merged rows:  {len(merged_df):,}')
print(f'rows lost:    {lost:,}')
print(
    'Rows are lost because some licensed devices (ACTIVE/PENDING_RENEWAL) '
    'have no matching entry in installed.json — the two datasets were '
    'likely produced at different times and do not have identical device coverage.'
)


merged rows:  43,297
rows lost:    0
Rows are lost because some licensed devices (ACTIVE/PENDING_RENEWAL) have no matching entry in installed.json — the two datasets were likely produced at different times and do not have identical device coverage.


## AND-102 Task 5: Merge 1 — Location Comparison and Filtering

In [6]:
# Both datasets carry a device location string ending in '<province> CA'.
# Extracting the two-letter province code (second-to-last token) gives a
# comparable, compact value that is robust to street-address differences.
def extract_province(s):
    if not isinstance(s, str):
        return None
    parts = s.strip().split()
    # Expected tail: ... <PROV> CA
    if len(parts) >= 2 and parts[-1] == 'CA':
        return parts[-2]
    return None

merged_df['loc_prov_license']   = merged_df['LocationoftheElevatingDevice'].apply(extract_province)
merged_df['loc_prov_installed'] = merged_df['Location of Device'].apply(extract_province)

mismatches = merged_df[merged_df['loc_prov_license'] != merged_df['loc_prov_installed']]
print(f'Rows with mismatched province: {len(mismatches):,}')
print(f'Rows with matching province:   {len(merged_df) - len(mismatches):,}')
print()
print('Mismatch sample:')
print(mismatches[['ElevatingDevicesNumber', 'LocationoftheElevatingDevice', 'Location of Device']].head(5).to_string())

Rows with mismatched province: 46
Rows with matching province:   43,251

Mismatch sample:
      ElevatingDevicesNumber LocationoftheElevatingDevice Location of Device
2586                   15251                          NaN                NaN
2697                   15474                          NaN                NaN
3855                   17166                          NaN                NaN
8542                   23274                          NaN                NaN
8544                   23278                          NaN                NaN


In [7]:
# Keep only rows where province matches between the two sources.
# "Match" means identical two-letter province codes — a disagreement signals
# the records may refer to different physical locations.
merged_df = merged_df[
    merged_df['loc_prov_license'] == merged_df['loc_prov_installed']
].drop(columns=['loc_prov_license', 'loc_prov_installed'])

print(f'Rows after location filter: {len(merged_df):,}  (dropped {43297 - len(merged_df):,} province-mismatched rows)')

Rows after location filter: 43,251  (dropped 46 province-mismatched rows)


## AND-102 Task 5: Merge 1 — Category Cleaning (Device Type)

In [8]:
print('Device Type — original categories:')
print(merged_df['Device Type'].value_counts().to_string())

# 'Freight Elevator-P' and 'Freight Elevator-E' are sub-classifications of
# Freight Elevator (P = personnel, E = electric). Collapsing them preserves
# the meaningful distinction between freight and passenger without fragmenting
# a category that has <30 total rows. 'Sidewalk Elevator' is mechanically a
# freight variant and is also folded in. 'Temporary Elevator', 'Special
# Installation', and 'Material Lift - ATD' (6 rows combined) don't fit
# cleanly into any main class, so they are grouped as 'Other'.
type_map = {
    'Freight Elevator-P': 'Freight Elevator',
    'Freight Elevator-E': 'Freight Elevator',
    'Sidewalk Elevator':  'Freight Elevator',
    'Temporary Elevator': 'Other',
    'Special Installation': 'Other',
    'Material Lift - ATD': 'Other',
}
merged_df['Device Type'] = merged_df['Device Type'].replace(type_map)

print()
print('Device Type — after cleaning:')
print(merged_df['Device Type'].value_counts().to_string())

Device Type — original categories:
Device Type
Passenger Elevator      39975
Freight Elevator         1763
LULA Elevator            1181
Observation Elevator      303
Freight Elevator-P         13
Freight Elevator-E          8
Temporary Elevator          4
Sidewalk Elevator           2
Special Installation        1
Material Lift - ATD         1


Device Type — after cleaning:
Device Type
Passenger Elevator      39975
Freight Elevator         1786
LULA Elevator            1181
Observation Elevator      303
Other                       6


## AND-102 Task 5: Merge 2 — Adding Alterations

In [9]:
altered_df = pd.read_json('../data/altered.json')
altered_df = altered_df.rename(columns={'Elevating Devices Number': 'ElevatingDevicesNumber'})

print(f'altered rows:              {len(altered_df):,}')
print(f'unique elevators altered:  {altered_df["ElevatingDevicesNumber"].nunique():,}')

# Aggregate to one row per elevator so the left join doesn't fan out our dataset.
alt_agg = (
    altered_df
    .groupby('ElevatingDevicesNumber')
    .agg(
        alteration_count=('ElevatingDevicesNumber', 'count'),
        latest_alteration_type=('Alteration Type', 'last'),
        latest_alteration_status=('Status of Alteration Request', 'last'),
    )
    .reset_index()
)

before = len(merged_df)
merged_df = merged_df.merge(alt_agg, on='ElevatingDevicesNumber', how='left')
print(f'\nRows before merge: {before:,}')
print(f'Rows after merge:  {len(merged_df):,}  (left join preserves all elevators; those never altered get NaN alteration fields)')
print(f'Elevators with no alteration record: {merged_df["alteration_count"].isna().sum():,}')

altered rows:              31,619
unique elevators altered:  22,340

Rows before merge: 43,251
Rows after merge:  43,251  (left join preserves all elevators; those never altered get NaN alteration fields)
Elevators with no alteration record: 21,212


In [10]:
many_alt = alt_agg[alt_agg['alteration_count'] >= 5]
pct = len(many_alt) / len(merged_df) * 100
print(f'Elevators with 5+ alterations: {len(many_alt):,} ({pct:.1f}% of the active fleet)')
print()
print('Top 10 most-altered elevators:')
print(many_alt.sort_values('alteration_count', ascending=False).head(10).to_string(index=False))

Elevators with 5+ alterations: 51 (0.1% of the active fleet)

Top 10 most-altered elevators:
 ElevatingDevicesNumber  alteration_count latest_alteration_type latest_alteration_status
                  14191                 7  ED-Minor B Alteration                     Open
                  61456                 7  ED-Minor B Alteration                   Passed
                  17604                 6    ED-Major Alteration                   Passed
                  28465                 6  ED-Minor B Alteration                     Open
                  23692                 6  ED-Minor B Alteration                   Passed
                  17922                 6  ED-Minor A Alteration                   Passed
                  23209                 6  ED-Minor B Alteration                   Closed
                  23693                 6  ED-Minor B Alteration                   Passed
                  23941                 6  ED-Minor B Alteration                   Passed
       

## AND-102 Task 5: Merge 3 — Adding Inspections

In [11]:
inspection_df = pd.read_csv('../data/inspection.csv')

# Explore the elevator-to-inspection relationship
insp_per_elev = inspection_df.groupby('ElevatingDevicesNumber')['InspectionNumber'].nunique()
elev_per_insp = inspection_df.groupby('InspectionNumber')['ElevatingDevicesNumber'].nunique()

print('Can one elevator have multiple inspections?')
print(f'  Max inspections per elevator: {insp_per_elev.max()}')
print(f'  Elevators with >1 inspection: {(insp_per_elev > 1).sum():,}')
print()
print('Can one inspection cover multiple elevators?')
print(f'  Max elevators per inspection: {elev_per_insp.max()}')
print(f'  Inspections covering >1 elevator: {(elev_per_insp > 1).sum():,}')
print()
print('Relationship: one elevator → many inspections (one-to-many).'
      ' Each inspection row covers exactly one elevator.')

Can one elevator have multiple inspections?
  Max inspections per elevator: 24
  Elevators with >1 inspection: 33,480

Can one inspection cover multiple elevators?
  Max elevators per inspection: 1
  Inspections covering >1 elevator: 0

Relationship: one elevator → many inspections (one-to-many). Each inspection row covers exactly one elevator.


In [12]:
# A naive merge would multiply rows (one per inspection per elevator).
# To keep the one-row-per-elevator structure we reduce inspection data to the
# most recent inspection record per elevator — the most operationally relevant.
inspection_df['Latest_INSPECTION_Date'] = pd.to_datetime(
    inspection_df['Latest_INSPECTION_Date'], errors='coerce'
)
latest_insp = (
    inspection_df
    .sort_values('Latest_INSPECTION_Date')
    .groupby('ElevatingDevicesNumber')
    .last()
    .reset_index()
)[['ElevatingDevicesNumber', 'InspectionType', 'Latest_INSPECTION_Date', 'InspectionOutcome']]

print(f'Inspection rows (raw):         {len(inspection_df):,}')
print(f'Unique elevators in inspection:{inspection_df["ElevatingDevicesNumber"].nunique():,}')
print(f'After keeping latest per elevator: {len(latest_insp):,} rows')

before = len(merged_df)
merged_df = merged_df.merge(latest_insp, on='ElevatingDevicesNumber', how='left')
print(f'\nRows before merge: {before:,}')
print(f'Rows after merge:  {len(merged_df):,}  (left join; elevators with no inspection record get NaN)')
print(f'Elevators with no inspection record: {merged_df["InspectionOutcome"].isna().sum():,}')
print(f'\nFinal columns ({len(merged_df.columns)}): {list(merged_df.columns)}')

Inspection rows (raw):         143,181
Unique elevators in inspection:40,954
After keeping latest per elevator: 40,954 rows

Rows before merge: 43,251
Rows after merge:  43,251  (left join; elevators with no inspection record get NaN)
Elevators with no inspection record: 3,429

Final columns (25): ['ElevatingDevicesNumber', 'LocationoftheElevatingDevice', 'ElevatingDevicesLicenseNumber', 'LICENSESTATUS', 'LICENSEEXPIRYDATE', 'LICENSEHOLDER', 'LICENSEHOLDERACCOUNTNUMBER', 'LICENSEHOLDERADDRESS', 'BILLINGCUSTOMER', 'BILLINGADDRESS', 'BILLINGACCOUNT', 'Owner Name', 'Owner Address', 'Owner Account Number', 'Device Class', 'Device Type', 'DeviceStatus', 'Location of Device', 'under review', 'alteration_count', 'latest_alteration_type', 'latest_alteration_status', 'InspectionType', 'Latest_INSPECTION_Date', 'InspectionOutcome']


In [13]:
merged_df.to_csv('../data/merged_elevator_data.csv', index=False)
print(f'Saved: data/merged_elevator_data.csv  ({len(merged_df):,} rows, {len(merged_df.columns)} columns)')

Saved: data/merged_elevator_data.csv  (43,251 rows, 25 columns)
